# 01 · Power Analysis and Experiment Design

How many users does an experiment need? Too few and real effects go undetected. Too many and you've burned time and opportunity cost.

This notebook derives the sample size formula, plots power curves, and runs simulations to confirm the math.

**Dataset:** Criteo retargeting experiment (~14M users). Primary metric: `visit` (site visit post-treatment, ~3.8% baseline rate). `Conversion` (0.2%) appears as a cautionary example.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Baseline rates from data

Only loading the columns needed. Full CSV is 3.1 GB, so `usecols` matters.

In [ ]:
df = pd.read_csv(
    'data/criteo-research-uplift-v2.1.csv',
    usecols=['treatment', 'visit', 'conversion']
)

ctrl  = df[df.treatment == 0]
treat = df[df.treatment == 1]

p0_visit = ctrl['visit'].mean()
p1_visit = treat['visit'].mean()
true_delta = p1_visit - p0_visit

p0_conv = ctrl['conversion'].mean()
p1_conv = treat['conversion'].mean()

n_ctrl  = len(ctrl)
n_treat = len(treat)
n_total = len(df)

print(f"{'':12s}  {'n':>10s}  {'visit rate':>10s}  {'conv rate':>10s}")
print(f"{'Control':12s}  {n_ctrl:>10,}  {p0_visit:>10.3%}  {p0_conv:>10.3%}")
print(f"{'Treatment':12s}  {n_treat:>10,}  {p1_visit:>10.3%}  {p1_conv:>10.3%}")
print(f"{'Lift':12s}  {'':>10s}  {true_delta:>+10.3%}  {p1_conv-p0_conv:>+10.3%}")
print(f"\nAllocation: {n_treat/n_total:.0%} treatment / {n_ctrl/n_total:.0%} control")

**A few things to notice:**

- Treatment/control split is 85/15, not 50/50. That costs power at fixed total N (quantified later).
- Visit rate (3.8% to 4.9%) is workable. The conversion lift (0.12pp on 0.19%) looks detectable at 60k users, but only because it's a 63% relative gain. At a more realistic 20% relative lift, the required sample jumps to 440k+. That's why downstream conversion usually forces a higher-funnel proxy metric.
- The primary metric has to be chosen before launch. Switching to one that "works" afterward is p-hacking.

## What is statistical power?

Two ways to be wrong in an experiment:

**Type I error (false positive, $\alpha$):** The treatment did nothing, but random noise looked like a lift. You ship a feature with zero effect. By convention $\alpha = 0.05$: you'd make this mistake 1 in 20 times across null experiments.

**Type II error (false negative, $\beta$):** The treatment worked, but the experiment was too small to see it. You kill something real. By convention $\beta = 0.20$.

**Power = $1 - \beta$.** If a real effect exists, power is the probability the test catches it. At 80% power, a real effect gets detected 4 out of 5 times.

**Minimum detectable effect (MDE):** The smallest lift worth detecting. This is a business question, not a statistics question. What's the minimum improvement that justifies shipping the feature? Answer with unit economics, not with whatever the formula can detect.

> $\alpha = 0.05$ and 80% power are 1930s academic conventions. The right thresholds depend on the cost of each error type. Shipping a harmful feature is a different failure mode from missing a small win.

## The sample size formula

For a two-sided test comparing two proportions $p_0$ (control) and $p_1 = p_0 + \delta$ (treatment), the required **per-group** sample size is:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot \bigl[p_0(1-p_0) + p_1(1-p_1)\bigr]}{\delta^2}$$

Read it as **noise / signal²**.

- Numerator: combined variance of both groups (the noise to see through).
- Denominator: the effect squared. Double the MDE and n drops by 4x.
- $z_{\alpha/2} = 1.96$ for $\alpha = 0.05$ (two-sided), $z_{\beta} = 0.84$ for 80% power. The sum $(1.96 + 0.84)^2 \approx 7.85$ is a constant you'll see cited in every sample size calculator.

This is an asymptotic approximation. It holds well when rates aren't near 0 or 1 and n is reasonable.

In [ ]:
def sample_size_per_group(p0, mde, alpha=0.05, power=0.80):
    """Required n per group for a two-sided two-proportion z-test."""
    p1 = p0 + mde
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(power)
    noise = p0 * (1 - p0) + p1 * (1 - p1)
    return int(np.ceil((z_a + z_b) ** 2 * noise / mde ** 2))


def theoretical_power(p0, mde, n, alpha=0.05):
    """Power for two-sided two-proportion z-test at n per group."""
    p1 = p0 + mde
    se = np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)) / n)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    # area to the right of the critical value under H1 (upper tail dominates)
    return float(stats.norm.cdf(abs(mde) / se - z_crit))


def empirical_power(p0, mde, n, alpha=0.05, n_sims=5_000):
    """Simulate experiments; return fraction that reject H0."""
    p1 = p0 + mde
    ctrl  = rng.binomial(n, p0, n_sims) / n
    treat = rng.binomial(n, p1, n_sims) / n
    pooled = (ctrl + treat) / 2
    se = np.sqrt(pooled * (1 - pooled) * 2 / n)
    z = (treat - ctrl) / se
    z_crit = stats.norm.ppf(1 - alpha / 2)
    return float((np.abs(z) > z_crit).mean())

## Apply to the Criteo numbers

In [ ]:
# How large would this experiment need to be to detect the observed lift?
n_needed = sample_size_per_group(p0_visit, true_delta)

print(f"Primary metric:   visit rate")
print(f"Baseline (p0):    {p0_visit:.3%}")
print(f"Observed lift:    {true_delta:+.3%} absolute  ({true_delta/p0_visit:.1%} relative)")
print(f"\nTo detect this lift at 80% power, α=0.05:")
print(f"  {n_needed:,} per group  →  {2*n_needed:,} total")
print(f"\nCriteo's actual control group: {n_ctrl:,}  ({n_ctrl/n_needed:.0f}× the minimum)")
print(f"Even the smaller group has enormous headroom at this effect size.")

print(f"\n--- Conversion rate comparison ---")
n_conv = sample_size_per_group(p0_conv, p1_conv - p0_conv)
rel_lift = (p1_conv - p0_conv) / p0_conv
n_conv_modest = sample_size_per_group(p0_conv, p0_conv * 0.20)  # 20% relative lift
print(f"Baseline (p0):    {p0_conv:.3%}")
print(f"Observed lift:    {p1_conv-p0_conv:+.3%}  ({rel_lift:.0%} relative)")
print(f"Required n/group: {n_conv:,}  →  {2*n_conv:,} total  (only feasible because the lift is enormous)")
print(f"At a more realistic 20% relative lift: {n_conv_modest:,}/group  →  {2*n_conv_modest:,} total")
print(f"Conversion as a primary metric typically forces you to use a higher-funnel proxy.")

## Power curves

Two questions every experiment designer needs to answer before launching:
1. **Given my traffic, what's the smallest effect I can reliably detect?**
2. **Given a target MDE, how long do I need to run the experiment?**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: power vs n for several MDEs
ns = np.arange(200, 25_001, 100)
for mde in [0.003, 0.005, 0.010, 0.015]:
    powers = [theoretical_power(p0_visit, mde, n) for n in ns]
    ax1.plot(ns, powers, label=f'MDE = {mde:.1%}')

ax1.axhline(0.80, color='gray', linestyle='--', linewidth=1, label='80% power')
ax1.set_xlabel('n per group')
ax1.set_ylabel('Power')
ax1.set_title('Power vs sample size\n(baseline visit rate 3.82%)')
ax1.set_ylim(0, 1)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax1.legend(fontsize=8)

# Right: required n vs MDE
mdes = np.linspace(0.003, 0.020, 200)
ns_required = [sample_size_per_group(p0_visit, d) for d in mdes]

ax2.plot(mdes * 100, ns_required, color='steelblue')
ax2.axvline(true_delta * 100, color='tomato', linestyle='--', linewidth=1.2,
            label=f'Observed lift ({true_delta:.2%})')
ax2.set_xlabel('MDE (percentage points)')
ax2.set_ylabel('Required n per group')
ax2.set_title('Required sample size vs MDE\n(80% power, α=0.05)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

# The quadratic relationship: cutting MDE in half quadruples required n
n_at_1pct  = sample_size_per_group(p0_visit, 0.010)
n_at_half  = sample_size_per_group(p0_visit, 0.005)
print(f"n for MDE=1.0pp: {n_at_1pct:,}")
print(f"n for MDE=0.5pp: {n_at_half:,}  ({n_at_half/n_at_1pct:.1f}× larger — MDE halved, n quadrupled)")

## The cost of unequal allocation

For fixed total N, 50/50 minimizes variance and maximizes power. Criteo's 85/15 split is common in ad campaigns (maximize ad exposure), but it has a real power cost.

Below: power as a function of allocation ratio, fixing N=10,000 total and using the observed visit lift as the target effect.

In [ ]:
total_n = 10_000
fractions = np.linspace(0.05, 0.95, 200)  # fraction assigned to treatment

powers_alloc = []
for f in fractions:
    n_t = max(1, int(total_n * f))
    n_c = max(1, total_n - n_t)
    # SE under alternative with unequal group sizes
    se = np.sqrt(p0_visit * (1 - p0_visit) / n_c + p1_visit * (1 - p1_visit) / n_t)
    z_crit = stats.norm.ppf(0.975)
    powers_alloc.append(stats.norm.cdf(true_delta / se - z_crit))

plt.figure(figsize=(8, 4))
plt.plot(fractions * 100, powers_alloc, color='steelblue')

criteo_f = n_treat / n_total
p_at_50  = powers_alloc[np.argmin(np.abs(fractions - 0.50))]
p_at_85  = powers_alloc[np.argmin(np.abs(fractions - criteo_f))]

plt.axvline(50, color='green', linestyle='--', linewidth=1.2,
            label=f'50/50 → power {p_at_50:.2f}')
plt.axvline(criteo_f * 100, color='tomato', linestyle='--', linewidth=1.2,
            label=f'{criteo_f:.0%}/{1-criteo_f:.0%} (Criteo) → power {p_at_85:.2f}')
plt.axhline(0.80, color='gray', linestyle=':', linewidth=1)

plt.xlabel('% of users in treatment group')
plt.ylabel(f'Power  (N={total_n:,} total, MDE={true_delta:.2%})')
plt.title('Power vs allocation ratio at fixed total sample size')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Power penalty from 50/50 → {criteo_f:.0%}/{1-criteo_f:.0%}: {p_at_50 - p_at_85:.2f} points")
print(f"At N=10k, the Criteo split would require a larger total N to compensate.")

## Simulation: does the formula hold?

The formula is an asymptotic approximation. Verify it by simulating many fake experiments and checking that the empirical rejection rate matches theoretical power.

Target: 1pp absolute lift against the 3.82% baseline. Two scenarios: adequate n (what the formula prescribes) vs 25% of that.

In [ ]:
mde_sim = 0.010  # 1pp absolute lift
n_adequate = sample_size_per_group(p0_visit, mde_sim)
n_under    = n_adequate // 4

theory_adequate = theoretical_power(p0_visit, mde_sim, n_adequate)
theory_under    = theoretical_power(p0_visit, mde_sim, n_under)

emp_adequate = empirical_power(p0_visit, mde_sim, n_adequate, n_sims=10_000)
emp_under    = empirical_power(p0_visit, mde_sim, n_under,    n_sims=10_000)

print(f"MDE = {mde_sim:.1%} absolute, baseline = {p0_visit:.3%}")
print(f"{'':30s}  {'n/group':>8s}  {'theory':>8s}  {'empirical':>10s}")
print(f"{'Adequate':30s}  {n_adequate:>8,}  {theory_adequate:>8.1%}  {emp_adequate:>10.1%}")
print(f"{'Underpowered (25% of n)':30s}  {n_under:>8,}  {theory_under:>8.1%}  {emp_under:>10.1%}")

In [ ]:
# Visualise: distribution of z-statistics under H1 for each scenario
n_sims = 5_000
p1_sim = p0_visit + mde_sim
z_crit = stats.norm.ppf(0.975)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, n, label in [
    (ax1, n_adequate, f'Adequately powered  (n={n_adequate:,}/group)'),
    (ax2, n_under,    f'Underpowered  (n={n_under:,}/group)'),
]:
    ctrl_s  = rng.binomial(n, p0_visit, n_sims) / n
    treat_s = rng.binomial(n, p1_sim,   n_sims) / n
    pooled  = (ctrl_s + treat_s) / 2
    se_s    = np.sqrt(pooled * (1 - pooled) * 2 / n)
    zs      = (treat_s - ctrl_s) / se_s

    detected = np.abs(zs) > z_crit
    ax.hist(zs[detected],  bins=50, alpha=0.7, color='tomato',    label=f'Rejected ({detected.mean():.1%})')
    ax.hist(zs[~detected], bins=50, alpha=0.7, color='steelblue', label=f'Missed   ({(~detected).mean():.1%})')
    ax.axvline( z_crit, color='black', linestyle='--', linewidth=1)
    ax.axvline(-z_crit, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('z-statistic')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle('z-statistic distribution under H1 (true effect = 1pp)', y=1.02)
plt.tight_layout()
plt.show()

## What this doesn't tell you

**You don't know $p_0$ upfront.** Historical conversion rates shift with season, traffic source, and product changes. When uncertain, use a lower baseline estimate. The cost is a larger n requirement; the alternative is arriving underpowered.

**MDE is a business question.** Teams often set it by asking "what's the smallest n we can afford?" and working backward. That's backwards. Start from business impact: what lift would actually justify shipping and maintaining the feature?

**One metric.** Testing 10 metrics simultaneously and calling any $p < 0.05$ a win pushes the effective false positive rate toward 40%. Pre-register one primary metric before launch.

**No peeking.** This framework assumes you collect all n users, then test once. Checking daily and stopping when $p < 0.05$ inflates Type I error substantially. Sequential testing (SPRT, always-valid inference) handles this; covered in notebook 02.

**IID.** The formula assumes users are independent. Network effects, day-of-week variation, novelty effects, and correlated traffic sources all violate this. Power analyses live in a cleaner world than real experiments do.